In [1]:
import json

import logomaker as lm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyliftover
import seaborn as sns
from pyfaidx import Fasta
from scipy.stats import chi2_contingency

from analysis_functions import (
                        calculate_chi2_p_values,
                        filter_and_convert_to_list,
                        check_ref,
                        get_codon_info,
                        get_context
                    )

## 1. Separate variants by pathogenicity value

Create dataframes for pathogenic/benign variants based on frequency.

* **pathogenic**  
 Cutoff in AC < 2. Additionally, intersect with the options in ClinVar and add pathogenic/likely pathogenic variants that are missing in GnomAD v.4, but are in ClinVar.

* **benign**  
 AC cut-off >= 2 (according to recent ACGS guidelines, BS2 criterion). In this case, we may have many autosomal recessive variants left, so let’s remove them. To do this, compare the resulting dataframe with benign ClinVar variants and remove all intersections with registered P/LP variants.

In [2]:
nmd_escape_df = pd.read_csv("data/lof_final_with_loeuf_pext_nmd_escape.csv")

In [3]:
nmd_escape_df.shape

(4426, 31)

In [4]:
pat_nmd_escape = nmd_escape_df.query('AC < 2')

In [5]:
pat_nmd_escape

,CHROM,POS,ID,REF,ALT,AC,Consequence,IMPACT,SYMBOL,Gene,...,FLAGS,VARIANT_CLASS,CANONICAL,LoF,LoF_filter,LoF_flags,LoF_info,LOEUF,pext,NMD_escape
1,chr1,2306243,rs1306616273,C,A,1,stop_gained,HIGH,SKI,ENSG00000157933,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.910379515317787,0.194,0.950938,YES
2,chr1,2306688,NaN,G,T,1,stop_gained,HIGH,SKI,ENSG00000157933,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.964791952446273,0.194,0.950938,YES
4,chr1,2306700,NaN,C,T,1,stop_gained,HIGH,SKI,ENSG00000157933,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.970278920896205,0.194,0.950938,YES
5,chr1,2306707,NaN,G,A,1,stop_gained,HIGH,SKI,ENSG00000157933,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.973479652491998,0.194,0.950938,YES
6,chr1,2306724,NaN,G,T,1,stop_gained,HIGH,SKI,ENSG00000157933,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.981252857796068,0.194,0.950938,YES
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4421,chr22,50275727,NaN,C,A,1,stop_gained,HIGH,PLXNB2,ENSG00000196576,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.995831067609208,0.285,1.000000,YES
4422,chr22,50447351,rs1556415963,T,A,1,stop_gained,HIGH,SBF1,ENSG00000100241,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.977472720872932,0.265,0.771752,YES
4423,chr22,50610725,NaN,C,G,1,stop_gained,HIGH,MAPK8IP2,ENSG00000008735,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.978181818181818,0.100,1.000000,YES
4424,chr22,50610732,NaN,G,T,1,stop_gained,HIGH,MAPK8IP2,ENSG00000008735,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.981010101010101,0.100,1.000000,YES


In [6]:
ben_nmd_escape = nmd_escape_df.query('AC >= 2')

In [7]:
ben_nmd_escape

,CHROM,POS,ID,REF,ALT,AC,Consequence,IMPACT,SYMBOL,Gene,...,FLAGS,VARIANT_CLASS,CANONICAL,LoF,LoF_filter,LoF_flags,LoF_info,LOEUF,pext,NMD_escape
0,chr1,2030133,NaN,G,T,7,stop_gained,HIGH,GABRD,ENSG00000187730,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.890360559234731,0.245,1.000000,YES
3,chr1,2306694,rs1375460797,C,T,3,stop_gained,HIGH,SKI,ENSG00000157933,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.967535436671239,0.194,0.950938,YES
11,chr1,6264626,rs745847219,G,A,15,stop_gained,HIGH,ACOT7,ENSG00000097021,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.973944294699012,0.193,0.994284,YES
12,chr1,9726957,rs1649753771,C,T,4,stop_gained,HIGH,PIK3CD,ENSG00000171608,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.971610845295056,0.196,0.855224,YES
15,chr1,9730654,NaN,C,A,6,stop_gained,HIGH,CLSTN1,ENSG00000171603,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.950441276306857,0.308,0.592226,YES
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4402,chr22,41178498,rs1354580969,C,T,3,stop_gained,HIGH,EP300,ENSG00000100393,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.936783988957902,0.099,1.000000,YES
4408,chr22,41178823,rs139208187,C,G,2,stop_gained,HIGH,EP300,ENSG00000100393,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.981642512077295,0.099,1.000000,YES
4410,chr22,41178931,rs1424813619,C,G,4,stop_gained,HIGH,EP300,ENSG00000100393,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.996549344375431,0.099,1.000000,YES
4413,chr22,41357424,NaN,G,T,2,stop_gained,HIGH,ZC3H7B,ENSG00000100403,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.998295841854124,0.323,0.988884,YES


In [8]:
ben_nmd_escape.columns

Index(['CHROM', 'POS', 'ID', 'REF', 'ALT', 'AC', 'Consequence', 'IMPACT',
       'SYMBOL', 'Gene', 'Feature', 'BIOTYPE', 'EXON', 'INTRON',
       'cDNA_position', 'CDS_position', 'Protein_position', 'Amino_acids',
       'Codons', 'ALLELE_NUM', 'STRAND', 'FLAGS', 'VARIANT_CLASS', 'CANONICAL',
       'LoF', 'LoF_filter', 'LoF_flags', 'LoF_info', 'LOEUF', 'pext',
       'NMD_escape'],
      dtype='object')

In [9]:
pat_nmd_escape['AC'].value_counts()  # 1    1669

AC
1    3105
Name: count, dtype: int64

In [10]:
ben_nmd_escape['AC'].value_counts()  # from 2 to 79

AC
2      587
3      212
4      142
5       87
6       58
      ... 
22       1
101      1
26       1
236      1
79       1
Name: count, Length: 62, dtype: int64

In [11]:
ben_nmd_escape['AC'].describe()

count    1321.000000
mean       14.320212
std       260.140981
min         2.000000
25%         2.000000
50%         3.000000
75%         5.000000
max      9426.000000
Name: AC, dtype: float64

### Remove all pathogenic Clinvar variants from benign dataframe

Merge `clinvar_nmd_esc_df` and `ben_nmd_escape` dataframes, remove all intersections by `CHROM`, `POS`, `REF`, `ALT`, and then remove the remainder of `clinvar_nmd_esc_df` (i.e. remove all rows that do not have an empty `CLNSIG` column).

In [12]:
clinvar_nmd_esc_df = pd.read_csv("data/clinvar_nmd_escape_df.csv")
# clinvar_nmd_esc = clinvar_nmd_esc_df.rename(columns={'Feature': 'Canonical_transcript'})

In [13]:
clinvar_nmd_esc_df

,CHROM,POS,ID,REF,ALT,CLNREVSTAT,CLNSIG,CLNVC,GENEINFO,MC,...,CDS_position,Protein_position,Amino_acids,Codons,STRAND,FLAGS,CANONICAL,LOEUF,pext,NMD_escape
0,chr1,12009699,2682761,C,A,"criteria_provided,_single_submitter",Likely_pathogenic,single_nucleotide_variant,MFN2:9927,SO:0001587|nonsense,...,2177.0,726.0,S/*,tCa/tAa,1.0,NaN,YES,0.282,0.971272,YES
1,chr1,12009699,933991,C,G,"criteria_provided,_single_submitter",Pathogenic,single_nucleotide_variant,MFN2:9927,SO:0001587|nonsense,...,2177.0,726.0,S/*,tCa/tGa,1.0,NaN,YES,0.282,0.971272,YES
2,chr1,12011511,1172816,G,A,"criteria_provided,_single_submitter",Likely_pathogenic,single_nucleotide_variant,MFN2:9927,SO:0001587|nonsense,...,2220.0,740.0,W/*,tgG/tgA,1.0,NaN,YES,0.282,0.971272,YES
3,chr1,12011542,572169,C,T,"criteria_provided,_multiple_submitters,_no_con...",Pathogenic,single_nucleotide_variant,MFN2:9927,SO:0001587|nonsense,...,2251.0,751.0,Q/*,Cag/Tag,1.0,NaN,YES,0.282,0.971272,YES
4,chr1,12011547,217162,C,A,"criteria_provided,_multiple_submitters,_no_con...",Pathogenic,single_nucleotide_variant,MFN2:9927,SO:0001587|nonsense,...,2256.0,752.0,Y/*,taC/taA,1.0,NaN,YES,0.282,0.971272,YES
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1223,chr22,41178165,2572086,C,T,"criteria_provided,_single_submitter",Likely_pathogenic,single_nucleotide_variant,EP300:2033,SO:0001587|nonsense,...,6454.0,2152.0,Q/*,Cag/Tag,1.0,NaN,YES,0.099,1.000000,YES
1224,chr22,41178243,2653214,C,T,"criteria_provided,_single_submitter",Likely_pathogenic,single_nucleotide_variant,EP300:2033,SO:0001587|nonsense,...,6532.0,2178.0,Q/*,Caa/Taa,1.0,NaN,YES,0.099,1.000000,YES
1225,chr22,41178498,2978296,C,T,"criteria_provided,_multiple_submitters,_no_con...",Pathogenic/Likely_pathogenic,single_nucleotide_variant,EP300:2033,SO:0001587|nonsense,...,6787.0,2263.0,R/*,Cga/Tga,1.0,NaN,YES,0.099,1.000000,YES
1226,chr22,41178579,451064,C,T,"criteria_provided,_single_submitter",Likely_pathogenic,single_nucleotide_variant,EP300:2033,SO:0001587|nonsense,...,6868.0,2290.0,Q/*,Cag/Tag,1.0,NaN,YES,0.099,1.000000,YES


In [14]:
clinvar_nmd_esc_df.columns

Index(['CHROM', 'POS', 'ID', 'REF', 'ALT', 'CLNREVSTAT', 'CLNSIG', 'CLNVC',
       'GENEINFO', 'MC', 'Consequence', 'SYMBOL', 'Gene', 'Feature_type',
       'Feature', 'BIOTYPE', 'cDNA_position', 'CDS_position',
       'Protein_position', 'Amino_acids', 'Codons', 'STRAND', 'FLAGS',
       'CANONICAL', 'LOEUF', 'pext', 'NMD_escape'],
      dtype='object')

In [15]:
merged_clinvar_and_ben = pd.concat([ben_nmd_escape, clinvar_nmd_esc_df], ignore_index=True)
merged_clinvar_and_ben

,CHROM,POS,ID,REF,ALT,AC,Consequence,IMPACT,SYMBOL,Gene,...,LoF_info,LOEUF,pext,NMD_escape,CLNREVSTAT,CLNSIG,CLNVC,GENEINFO,MC,Feature_type
0,chr1,2030133,NaN,G,T,7.0,stop_gained,HIGH,GABRD,ENSG00000187730,...,PERCENTILE:0.890360559234731,0.245,1.000000,YES,NaN,NaN,NaN,NaN,NaN,NaN
1,chr1,2306694,rs1375460797,C,T,3.0,stop_gained,HIGH,SKI,ENSG00000157933,...,PERCENTILE:0.967535436671239,0.194,0.950938,YES,NaN,NaN,NaN,NaN,NaN,NaN
2,chr1,6264626,rs745847219,G,A,15.0,stop_gained,HIGH,ACOT7,ENSG00000097021,...,PERCENTILE:0.973944294699012,0.193,0.994284,YES,NaN,NaN,NaN,NaN,NaN,NaN
3,chr1,9726957,rs1649753771,C,T,4.0,stop_gained,HIGH,PIK3CD,ENSG00000171608,...,PERCENTILE:0.971610845295056,0.196,0.855224,YES,NaN,NaN,NaN,NaN,NaN,NaN
4,chr1,9730654,NaN,C,A,6.0,stop_gained,HIGH,CLSTN1,ENSG00000171603,...,PERCENTILE:0.950441276306857,0.308,0.592226,YES,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2544,chr22,41178165,2572086,C,T,NaN,stop_gained,NaN,EP300,ENSG00000100393,...,NaN,0.099,1.000000,YES,"criteria_provided,_single_submitter",Likely_pathogenic,single_nucleotide_variant,EP300:2033,SO:0001587|nonsense,Transcript
2545,chr22,41178243,2653214,C,T,NaN,stop_gained,NaN,EP300,ENSG00000100393,...,NaN,0.099,1.000000,YES,"criteria_provided,_single_submitter",Likely_pathogenic,single_nucleotide_variant,EP300:2033,SO:0001587|nonsense,Transcript
2546,chr22,41178498,2978296,C,T,NaN,stop_gained,NaN,EP300,ENSG00000100393,...,NaN,0.099,1.000000,YES,"criteria_provided,_multiple_submitters,_no_con...",Pathogenic/Likely_pathogenic,single_nucleotide_variant,EP300:2033,SO:0001587|nonsense,Transcript
2547,chr22,41178579,451064,C,T,NaN,stop_gained,NaN,EP300,ENSG00000100393,...,NaN,0.099,1.000000,YES,"criteria_provided,_single_submitter",Likely_pathogenic,single_nucleotide_variant,EP300:2033,SO:0001587|nonsense,Transcript


In [16]:
merged_clinvar_and_ben.columns

Index(['CHROM', 'POS', 'ID', 'REF', 'ALT', 'AC', 'Consequence', 'IMPACT',
       'SYMBOL', 'Gene', 'Feature', 'BIOTYPE', 'EXON', 'INTRON',
       'cDNA_position', 'CDS_position', 'Protein_position', 'Amino_acids',
       'Codons', 'ALLELE_NUM', 'STRAND', 'FLAGS', 'VARIANT_CLASS', 'CANONICAL',
       'LoF', 'LoF_filter', 'LoF_flags', 'LoF_info', 'LOEUF', 'pext',
       'NMD_escape', 'CLNREVSTAT', 'CLNSIG', 'CLNVC', 'GENEINFO', 'MC',
       'Feature_type'],
      dtype='object')

In [17]:
# remove duplicates
ben_nmd_escape_filtered = merged_clinvar_and_ben.drop_duplicates(subset=['CHROM', 'POS', 'REF', 'ALT'], keep=False)

In [18]:
# remove the Clinvar df
ben_nmd_escape_filtered = ben_nmd_escape_filtered[~ben_nmd_escape_filtered['CLNSIG'].notna()]

In [19]:
ben_nmd_escape_filtered.columns

Index(['CHROM', 'POS', 'ID', 'REF', 'ALT', 'AC', 'Consequence', 'IMPACT',
       'SYMBOL', 'Gene', 'Feature', 'BIOTYPE', 'EXON', 'INTRON',
       'cDNA_position', 'CDS_position', 'Protein_position', 'Amino_acids',
       'Codons', 'ALLELE_NUM', 'STRAND', 'FLAGS', 'VARIANT_CLASS', 'CANONICAL',
       'LoF', 'LoF_filter', 'LoF_flags', 'LoF_info', 'LOEUF', 'pext',
       'NMD_escape', 'CLNREVSTAT', 'CLNSIG', 'CLNVC', 'GENEINFO', 'MC',
       'Feature_type'],
      dtype='object')

In [20]:
pat_nmd_escape.columns

Index(['CHROM', 'POS', 'ID', 'REF', 'ALT', 'AC', 'Consequence', 'IMPACT',
       'SYMBOL', 'Gene', 'Feature', 'BIOTYPE', 'EXON', 'INTRON',
       'cDNA_position', 'CDS_position', 'Protein_position', 'Amino_acids',
       'Codons', 'ALLELE_NUM', 'STRAND', 'FLAGS', 'VARIANT_CLASS', 'CANONICAL',
       'LoF', 'LoF_filter', 'LoF_flags', 'LoF_info', 'LOEUF', 'pext',
       'NMD_escape'],
      dtype='object')

Remove unnecessary columns left after Clinvar.

In [21]:
# ben_nmd_escape_filtered = ben_nmd_escape_filtered.drop(columns=['ID', 'CLNSIG', 'CLNVC', 'GENEINFO', 'CLNREVSTAT', 'MC', 'SYMBOL', 
#                                                 'Gene', 'Feature_type', 'BIOTYPE', 'CANONICAL'])
ben_nmd_escape_filtered = ben_nmd_escape_filtered.drop(columns=['CLNSIG', 'CLNVC', 'GENEINFO', 'MC', 'CLNREVSTAT', 'Feature_type'])
ben_nmd_escape_filtered

,CHROM,POS,ID,REF,ALT,AC,Consequence,IMPACT,SYMBOL,Gene,...,FLAGS,VARIANT_CLASS,CANONICAL,LoF,LoF_filter,LoF_flags,LoF_info,LOEUF,pext,NMD_escape
0,chr1,2030133,NaN,G,T,7.0,stop_gained,HIGH,GABRD,ENSG00000187730,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.890360559234731,0.245,1.000000,YES
1,chr1,2306694,rs1375460797,C,T,3.0,stop_gained,HIGH,SKI,ENSG00000157933,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.967535436671239,0.194,0.950938,YES
2,chr1,6264626,rs745847219,G,A,15.0,stop_gained,HIGH,ACOT7,ENSG00000097021,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.973944294699012,0.193,0.994284,YES
3,chr1,9726957,rs1649753771,C,T,4.0,stop_gained,HIGH,PIK3CD,ENSG00000171608,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.971610845295056,0.196,0.855224,YES
4,chr1,9730654,NaN,C,A,6.0,stop_gained,HIGH,CLSTN1,ENSG00000171603,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.950441276306857,0.308,0.592226,YES
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1315,chr22,41178264,NaN,C,T,3.0,stop_gained,HIGH,EP300,ENSG00000100393,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.904485852311939,0.099,1.000000,YES
1317,chr22,41178823,rs139208187,C,G,2.0,stop_gained,HIGH,EP300,ENSG00000100393,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.981642512077295,0.099,1.000000,YES
1318,chr22,41178931,rs1424813619,C,G,4.0,stop_gained,HIGH,EP300,ENSG00000100393,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.996549344375431,0.099,1.000000,YES
1319,chr22,41357424,NaN,G,T,2.0,stop_gained,HIGH,ZC3H7B,ENSG00000100403,...,NaN,SNV,YES,HC,NaN,NaN,PERCENTILE:0.998295841854124,0.323,0.988884,YES


In [22]:
ben_nmd_escape_filtered.columns == pat_nmd_escape.columns

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True])

In [23]:
ben_nmd_escape_filtered.shape

(1290, 31)

Before filtering:

In [24]:
ben_nmd_escape.shape

(1321, 31)

**The dataframe with benign variants is ready.**

In [25]:
pat_nmd_escape_filtered = pat_nmd_escape.copy()

**The dataframe with pathogenic variants is ready.**

### Add 5th group - Clinvar Pathogenic

In [26]:
clinvar_escape_df = pd.read_csv("data/clinvar_nmd_escape_df.csv")

In [27]:
clinvar_escape_df.columns

Index(['CHROM', 'POS', 'ID', 'REF', 'ALT', 'CLNREVSTAT', 'CLNSIG', 'CLNVC',
       'GENEINFO', 'MC', 'Consequence', 'SYMBOL', 'Gene', 'Feature_type',
       'Feature', 'BIOTYPE', 'cDNA_position', 'CDS_position',
       'Protein_position', 'Amino_acids', 'Codons', 'STRAND', 'FLAGS',
       'CANONICAL', 'LOEUF', 'pext', 'NMD_escape'],
      dtype='object')

In [28]:
clinvar_escape_df['Gene'].value_counts()

Gene
ENSG00000134982    380
ENSG00000176165     48
ENSG00000096696     44
ENSG00000185129     35
ENSG00000141431     29
                  ... 
ENSG00000178951      1
ENSG00000142156      1
ENSG00000087460      1
ENSG00000101115      1
ENSG00000100345      1
Name: count, Length: 169, dtype: int64

## 2. Balance dataframes

In both dataframes, we will leave only those genes that are found in both `ben_nmd_escape_filtered` and `pat_nmd_escape_filtered`, and also equalize the number of variants in each gene.

In [29]:
unique_genes_ben = ben_nmd_escape_filtered['SYMBOL'].unique()
unique_genes_pat = pat_nmd_escape_filtered['SYMBOL'].unique()
unique_genes_clinvar = clinvar_escape_df['SYMBOL'].unique()

# Находим пересечение сначала между двумя массивами, затем с третьим
intersected_genes = np.intersect1d(np.intersect1d(unique_genes_pat, unique_genes_ben), unique_genes_clinvar)

print("Unique genes in pat dataframe:", len(unique_genes_pat))
print("Unique genes in ben dataframe:", len(unique_genes_ben))
print("Unique genes in clinvar dataframe:", len(unique_genes_clinvar))
print("Intersected genes:", len(intersected_genes))


Unique genes in pat dataframe: 1183
Unique genes in ben dataframe: 758
Unique genes in clinvar dataframe: 169
Intersected genes: 50


In [30]:
common_genes = set(unique_genes_ben) & set(unique_genes_pat) & set(unique_genes_clinvar)

ben_nmd_escape_filtered = ben_nmd_escape_filtered[ben_nmd_escape_filtered['SYMBOL'].isin(common_genes)]
pat_nmd_escape_filtered = pat_nmd_escape_filtered[pat_nmd_escape_filtered['SYMBOL'].isin(common_genes)]
clinvar_escape_df = clinvar_escape_df[clinvar_escape_df['SYMBOL'].isin(common_genes)]

In [31]:
# Проверка
unique_genes_ben = ben_nmd_escape_filtered['SYMBOL'].unique()
unique_genes_pat = pat_nmd_escape_filtered['SYMBOL'].unique()
unique_genes_clinvar = clinvar_escape_df['SYMBOL'].unique()

intersected_genes = np.intersect1d(np.intersect1d(unique_genes_pat, unique_genes_ben), unique_genes_clinvar)

print("Unique genes in pat dataframe:", len(unique_genes_pat))
print("Unique genes in ben dataframe:", len(unique_genes_ben))
print("Unique genes in clinvar dataframe:", len(unique_genes_clinvar))
print("Intersected genes:", len(intersected_genes))

Unique genes in pat dataframe: 50
Unique genes in ben dataframe: 50
Unique genes in clinvar dataframe: 50
Intersected genes: 50


In [32]:
count_pat = pat_nmd_escape_filtered['SYMBOL'].value_counts()
count_ben = ben_nmd_escape_filtered['SYMBOL'].value_counts()
count_clinvar = clinvar_escape_df['SYMBOL'].value_counts()

min_counts = pd.concat([count_pat, count_ben, count_clinvar], axis=1).min(axis=1)

pat_nmd_escape_final = pd.concat([
    pat_nmd_escape_filtered[pat_nmd_escape_filtered['SYMBOL'] == gene].sample(n=min_count, random_state=42) \
    for gene, min_count in min_counts.items()
])

ben_nmd_escape_final = pd.concat([
    ben_nmd_escape_filtered[ben_nmd_escape_filtered['SYMBOL'] == gene].sample(n=min_count, random_state=42) \
    for gene, min_count in min_counts.items()
])

clinvar_escape_df = pd.concat([
    clinvar_escape_df[clinvar_escape_df['SYMBOL'] == gene].sample(n=min_count, random_state=42) \
    for gene, min_count in min_counts.items()
])

In [33]:
pat_nmd_escape_final.shape == ben_nmd_escape_final.shape

True

In [34]:
pat_nmd_escape_final.shape

(77, 31)

In [35]:
ben_nmd_escape_final.shape

(77, 31)

In [36]:
pat_nmd_escape_final['SYMBOL'].nunique() == ben_nmd_escape_final['SYMBOL'].nunique()

True

In [37]:
pat_nmd_escape_final['SYMBOL'].value_counts().sum() == ben_nmd_escape_final['SYMBOL'].value_counts().sum()

np.True_

In [38]:
pat_nmd_escape_final['SYMBOL'].nunique()

50

In [39]:
ben_nmd_escape_final['SYMBOL'].nunique()

50

In [40]:
pat_nmd_escape_final.shape

(77, 31)

In [41]:
ben_nmd_escape_final.shape

(77, 31)

Dataframes are now balanced by genes and number of variants.

## 3. Get sequence context

Write the context in the corresponding column of the dataframe.

File `gencode.v47.transcripts.fa.gz` can be downloaded from the GENCODE database (https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_47/gencode.v47.transcripts.fa.gz).

In [42]:
ben_nmd_escape_final.dtypes

CHROM                object
POS                   int64
ID                   object
REF                  object
ALT                  object
AC                  float64
Consequence          object
IMPACT               object
SYMBOL               object
Gene                 object
Feature              object
BIOTYPE              object
EXON                 object
INTRON              float64
cDNA_position       float64
CDS_position        float64
Protein_position    float64
Amino_acids          object
Codons               object
ALLELE_NUM          float64
STRAND              float64
FLAGS               float64
VARIANT_CLASS        object
CANONICAL            object
LoF                  object
LoF_filter          float64
LoF_flags           float64
LoF_info             object
LOEUF               float64
pext                float64
NMD_escape           object
dtype: object

In [43]:
ben_nmd_escape_final['Codons']

390     tCa/tAa
442     Cga/Tga
447     taC/taA
443     Caa/Taa
440     Cag/Tag
         ...   
609     Caa/Taa
532     taC/taG
533     Cag/Tag
1133    tgG/tgA
1233    Cga/Tga
Name: Codons, Length: 77, dtype: object

In [44]:
ben_nmd_escape_final['cDNA_position'] = ben_nmd_escape_final['cDNA_position'].astype(int)

In [45]:
pat_nmd_escape_final['cDNA_position'] = pat_nmd_escape_final['cDNA_position'].astype(int)

In [46]:
clinvar_escape_df['cDNA_position'] = clinvar_escape_df['cDNA_position'].astype(int)

In [47]:
transcript_fasta = Fasta("data/gencode.v47.transcripts.fa.gz", key_function = lambda x: x.split('.')[0])

In [48]:
get_context(ben_nmd_escape_final, transcript_fasta, 13, 12)

'Contexts have been added to the dataframe!'

In [49]:
get_context(pat_nmd_escape_final, transcript_fasta, 13, 12)

'Contexts have been added to the dataframe!'

In [50]:
get_context(clinvar_escape_df, transcript_fasta, 13, 12)

'Contexts have been added to the dataframe!'

In [51]:
pat_nmd_escape_final.iloc[1]

CHROM                                       chr6
POS                                      7584770
ID                                           NaN
REF                                            G
ALT                                            A
AC                                             1
Consequence                          stop_gained
IMPACT                                      HIGH
SYMBOL                                       DSP
Gene                             ENSG00000096696
Feature                          ENST00000379802
BIOTYPE                           protein_coding
EXON                                       24/24
INTRON                                       NaN
cDNA_position                               7753
CDS_position                              7508.0
Protein_position                          2503.0
Amino_acids                                  W/*
Codons                                   tGg/tAg
ALLELE_NUM                                     1
STRAND              

## 4. Get variant codons information

In [52]:
ben_nmd_escape_final['Codons']

390     tCa/tAa
442     Cga/Tga
447     taC/taA
443     Caa/Taa
440     Cag/Tag
         ...   
609     Caa/Taa
532     taC/taG
533     Cag/Tag
1133    tgG/tgA
1233    Cga/Tga
Name: Codons, Length: 77, dtype: object

In [53]:
def get_codon_position(codon_change):
    codon = codon_change.split('/')[0]  # первый кодон (до слэша)
    for i, base in enumerate(codon, start=1):
        if base.isupper():
            return i
    return None  # если вдруг нет заглавной буквы

In [54]:
ben_nmd_escape_final['Codon_position'] = ben_nmd_escape_final['Codons'].apply(get_codon_position)

In [55]:
ben_nmd_escape_final['Codon_position'].value_counts()

Codon_position
1    53
2    16
3     8
Name: count, dtype: int64

In [56]:
pat_nmd_escape_final['Codon_position'] = pat_nmd_escape_final['Codons'].apply(get_codon_position)

In [57]:
pat_nmd_escape_final['Codon_position'].value_counts()

Codon_position
1    48
3    15
2    14
Name: count, dtype: int64

In [58]:
clinvar_escape_df['Codon_position'] = clinvar_escape_df['Codons'].apply(get_codon_position)

In [59]:
clinvar_escape_df['Codon_position'].value_counts()

Codon_position
1    56
3    17
2     4
Name: count, dtype: int64

In [60]:
def get_initial_codon(codon_change):
    return codon_change.split('/')[0].upper()

def get_stop_codon(codon_change):
    return codon_change.split('/')[1].upper()

In [61]:
ben_nmd_escape_final['Initial_Codon'] = ben_nmd_escape_final['Codons'].apply(get_initial_codon)
ben_nmd_escape_final['Stop_Codon'] = ben_nmd_escape_final['Codons'].apply(get_stop_codon)

In [62]:
pat_nmd_escape_final['Initial_Codon'] = pat_nmd_escape_final['Codons'].apply(get_initial_codon)
pat_nmd_escape_final['Stop_Codon'] = pat_nmd_escape_final['Codons'].apply(get_stop_codon)

In [63]:
clinvar_escape_df['Initial_Codon'] = clinvar_escape_df['Codons'].apply(get_initial_codon)
clinvar_escape_df['Stop_Codon'] = clinvar_escape_df['Codons'].apply(get_stop_codon)

In [64]:
ben_nmd_escape_final['Codon_change'] = list(zip(ben_nmd_escape_final['Initial_Codon'], ben_nmd_escape_final['Stop_Codon']))
ben_nmd_escape_final['Codon_change'].value_counts(normalize=True) * 100

Codon_change
(CGA, TGA)    31.168831
(CAG, TAG)    16.883117
(TCA, TGA)     9.090909
(CAA, TAA)     6.493506
(TGG, TAG)     6.493506
(TCA, TAA)     5.194805
(GAA, TAA)     5.194805
(GAG, TAG)     5.194805
(TGG, TGA)     5.194805
(TAC, TAG)     2.597403
(GGA, TGA)     2.597403
(TAC, TAA)     1.298701
(AAG, TAG)     1.298701
(TAT, TAA)     1.298701
Name: proportion, dtype: float64

In [65]:
pat_nmd_escape_final['Codon_change'] = list(zip(pat_nmd_escape_final['Initial_Codon'], pat_nmd_escape_final['Stop_Codon']))
pat_nmd_escape_final['Codon_change'].value_counts(normalize=True) * 100

Codon_change
(CAG, TAG)    23.376623
(CGA, TGA)    11.688312
(CAA, TAA)    10.389610
(TCA, TGA)     7.792208
(GAA, TAA)     6.493506
(GAG, TAG)     6.493506
(TAC, TAG)     5.194805
(TGG, TAG)     3.896104
(TAT, TAA)     3.896104
(TAC, TAA)     3.896104
(TGG, TGA)     3.896104
(TCA, TAA)     2.597403
(TTA, TGA)     2.597403
(TGC, TGA)     2.597403
(GGA, TGA)     1.298701
(AAA, TAA)     1.298701
(TCG, TAG)     1.298701
(AAG, TAG)     1.298701
Name: proportion, dtype: float64

In [66]:
clinvar_escape_df['Codon_change'] = list(zip(clinvar_escape_df['Initial_Codon'], clinvar_escape_df['Stop_Codon']))
clinvar_escape_df['Codon_change'].value_counts(normalize=True) * 100

Codon_change
(CAG, TAG)    22.077922
(CGA, TGA)    19.480519
(TAC, TAG)     6.493506
(GAA, TAA)     6.493506
(AAG, TAG)     6.493506
(GAG, TAG)     6.493506
(TGG, TGA)     5.194805
(CAA, TAA)     5.194805
(TAT, TAG)     3.896104
(TCA, TGA)     3.896104
(TAC, TAA)     3.896104
(AAA, TAA)     2.597403
(GGA, TGA)     2.597403
(TGC, TGA)     2.597403
(AGA, TGA)     1.298701
(TCG, TAG)     1.298701
Name: proportion, dtype: float64

## 8. Relationship between the significance of a variant and its position in a codon

In [67]:
pat_nmd_escape_final.loc[:, 'Significance'] = 'pathogenic'
ben_nmd_escape_final.loc[:, 'Significance'] = 'benign'

In [68]:
all_nmd_escape_final = pd.concat([pat_nmd_escape_final, ben_nmd_escape_final], ignore_index=True)

In [69]:
all_nmd_escape_final = all_nmd_escape_final.loc[all_nmd_escape_final['Codon_position'] != 'No_stop']

In [70]:
all_nmd_escape_final.to_csv('data/2A-i_all_nmd_escape_final_wo_in_clinvar.csv', index=False)

In [71]:
clinvar_escape_df.to_csv('data/2A-i_clinvar_nmd_escape_final.csv', index=False)

In [72]:
clinvar_escape_df

,CHROM,POS,ID,REF,ALT,CLNREVSTAT,CLNSIG,CLNVC,GENEINFO,MC,...,FLAGS,CANONICAL,LOEUF,pext,NMD_escape,Context,Codon_position,Initial_Codon,Stop_Codon,Codon_change
467,chr5,112840390,469977,C,G,"criteria_provided,_multiple_submitters,_no_con...",Pathogenic,single_nucleotide_variant,APC:324,SO:0001587|nonsense,...,NaN,YES,0.161,0.904855,YES,CCCAGACTGCTTCAAAATTACCTCC,2,TCA,TGA,"(TCA, TGA)"
688,chr6,7585091,981513,C,G,"criteria_provided,_single_submitter",Likely_pathogenic,single_nucleotide_variant,DSP:1832,SO:0001587|nonsense,...,NaN,YES,0.260,0.999478,YES,GCAGCTCTTTTTCAGACACCCTGGA,2,TCA,TGA,"(TCA, TGA)"
675,chr6,7584112,199905,C,T,"criteria_provided,_multiple_submitters,_no_con...",Pathogenic/Likely_pathogenic,single_nucleotide_variant,DSP:1832,SO:0001587|nonsense,...,NaN,YES,0.260,0.999478,YES,ATTGGCTTAGTCCGACCTGGTACTG,1,CGA,TGA,"(CGA, TGA)"
676,chr6,7584202,372958,G,T,"criteria_provided,_single_submitter",Pathogenic,single_nucleotide_variant,DSP:1832,SO:0001587|nonsense,...,NaN,YES,0.260,0.999478,YES,TTACCAGTGGAGGAAGCCTACAAGA,1,GAA,TAA,"(GAA, TAA)"
687,chr6,7585018,2581755,C,T,"criteria_provided,_multiple_submitters,_no_con...",Pathogenic,single_nucleotide_variant,DSP:1832,SO:0001587|nonsense,...,NaN,YES,0.260,0.999478,YES,TTTAGCAGCTCCCGACATGAATCAG,1,CGA,TGA,"(CGA, TGA)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
815,chr8,115414124,420782,G,A,"criteria_provided,_single_submitter",Likely_pathogenic,single_nucleotide_variant,TRPS1:7227,SO:0001587|nonsense,...,NaN,YES,0.111,0.870956,YES,TGCAGCATATGCCAGCATCTTTGCA,1,CAG,TAG,"(CAG, TAG)"
723,chr7,35204518,2006880,C,A,"criteria_provided,_single_submitter",Pathogenic,single_nucleotide_variant,TBX20:57057,SO:0001587|nonsense,...,NaN,YES,0.315,1.000000,YES,CGTACCTACGGAGGAGAAGAAGATG,1,GGA,TGA,"(GGA, TGA)"
733,chr7,41965334,2662836,G,A,"criteria_provided,_single_submitter",Likely_pathogenic,single_nucleotide_variant,GLI3:2737,SO:0001587|nonsense,...,NaN,YES,0.195,0.898291,YES,GCCTTCCATGAACAGCCCTGTAAGG,1,CAG,TAG,"(CAG, TAG)"
1108,chr17,72122970,2040173,C,A,"criteria_provided,_single_submitter",Pathogenic,single_nucleotide_variant,SOX9:6662,SO:0001587|nonsense,...,NaN,YES,0.168,1.000000,YES,CCGGCGAGCACTCGGGGCAATCCCA,2,TCG,TAG,"(TCG, TAG)"


In [73]:
pat_nmd_escape_final

,CHROM,POS,ID,REF,ALT,AC,Consequence,IMPACT,SYMBOL,Gene,...,LoF_info,LOEUF,pext,NMD_escape,Context,Codon_position,Initial_Codon,Stop_Codon,Codon_change,Significance
1214,chr5,112839436,rs1765536013,C,A,1,stop_gained,HIGH,APC,ENSG00000134982,...,PERCENTILE:0.450304735114862,0.161,0.904855,YES,TATCATCTTTGTCATCAGCTGAAGA,2,TCA,TAA,"(TCA, TAA)",pathogenic
1388,chr6,7584770,NaN,G,A,1,stop_gained,HIGH,DSP,ENSG00000096696,...,PERCENTILE:0.871402042711235,0.260,0.999478,YES,AGGAATGTGAATGGGAAGAAATAAC,2,TGG,TAG,"(TGG, TAG)",pathogenic
1376,chr6,7583378,NaN,T,G,1,stop_gained,HIGH,DSP,ENSG00000096696,...,PERCENTILE:0.709842154131848,0.260,0.999478,YES,AGAGAAAGAAATTAATCAGCCCAGA,2,TTA,TGA,"(TTA, TGA)",pathogenic
1366,chr6,7582693,rs2113697854,G,T,1,stop_gained,HIGH,DSP,ENSG00000096696,...,PERCENTILE:0.630338904363974,0.260,0.999478,YES,CAGGTGGTACAGGAAAGAGAGAGCC,1,GAA,TAA,"(GAA, TAA)",pathogenic
1375,chr6,7583287,NaN,C,T,1,stop_gained,HIGH,DSP,ENSG00000096696,...,PERCENTILE:0.699280408542247,0.260,0.999478,YES,GCTTCTGAAATCCAGCCATTCCTTC,1,CAG,TAG,"(CAG, TAG)",pathogenic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2003,chr8,115414918,rs763363982,G,C,1,stop_gained,HIGH,TRPS1,ENSG00000104447,...,PERCENTILE:0.76962676962677,0.111,0.997035,YES,TAGAGAGGAGGTCAGAAGATCATCT,2,TCA,TGA,"(TCA, TGA)",pathogenic
1695,chr7,35202516,rs377570351,G,A,1,stop_gained,HIGH,TBX20,ENSG00000164532,...,PERCENTILE:0.936011904761905,0.315,1.000000,YES,TTCCACATGCCGCGATACCATCACT,1,CGA,TGA,"(CGA, TGA)",pathogenic
1700,chr7,41964335,NaN,G,A,1,stop_gained,HIGH,GLI3,ENSG00000106571,...,PERCENTILE:0.998945814885094,0.195,0.898291,YES,CTTGCAGTTATGCAATAGGCTTTAG,1,CAA,TAA,"(CAA, TAA)",pathogenic
3789,chr17,72124037,rs1057518216,C,T,1,stop_gained,HIGH,SOX9,ENSG00000125398,...,PERCENTILE:0.77124183006536,0.168,1.000000,YES,GGCCAGTCCCAGCGAACGCACATCA,1,CGA,TGA,"(CGA, TGA)",pathogenic


In [74]:
ben_nmd_escape_final

,CHROM,POS,ID,REF,ALT,AC,Consequence,IMPACT,SYMBOL,Gene,...,LoF_info,LOEUF,pext,NMD_escape,Context,Codon_position,Initial_Codon,Stop_Codon,Codon_change,Significance
390,chr5,112841659,rs1554087174,C,A,2.0,stop_gained,HIGH,APC,ENSG00000134982,...,PERCENTILE:0.710853258321613,0.161,0.904855,YES,CAGTTTGTTTCTCAAGAAACAGTTC,2,TCA,TAA,"(TCA, TAA)",benign
442,chr6,7584262,rs369482721,C,T,3.0,stop_gained,HIGH,DSP,ENSG00000096696,...,PERCENTILE:0.812441968430826,0.260,0.999478,YES,CTGTCTGCAGAACGAGCTGTCACTG,1,CGA,TGA,"(CGA, TGA)",benign
447,chr6,7585713,NaN,C,A,2.0,stop_gained,HIGH,DSP,ENSG00000096696,...,PERCENTILE:0.980849582172702,0.260,0.999478,YES,ACCCAGCCCTTACAACATGTCTTCG,3,TAC,TAA,"(TAC, TAA)",benign
443,chr6,7584631,NaN,C,T,2.0,stop_gained,HIGH,DSP,ENSG00000096696,...,PERCENTILE:0.855269266480966,0.260,0.999478,YES,GTGCAGACATCACAAAAGAATACCC,1,CAA,TAA,"(CAA, TAA)",benign
440,chr6,7583872,rs876657798,C,T,2.0,stop_gained,HIGH,DSP,ENSG00000096696,...,PERCENTILE:0.767177344475395,0.260,0.999478,YES,TTGCTTTCAGTACAGAAGAGAAGCA,1,CAG,TAG,"(CAG, TAG)",benign
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
609,chr8,115414055,rs759477744,G,A,4.0,stop_gained,HIGH,TRPS1,ENSG00000104447,...,PERCENTILE:0.991763191763192,0.111,0.870956,YES,AGGAACAATGCACAAGTGGAAAAAA,1,CAA,TAA,"(CAA, TAA)",benign
532,chr7,35204522,rs113335362,G,C,3.0,stop_gained,HIGH,TBX20,ENSG00000164532,...,PERCENTILE:0.707589285714286,0.315,1.000000,YES,CATCCGTACCTACGGAGGAGAAGAA,3,TAC,TAG,"(TAC, TAG)",benign
533,chr7,41965049,NaN,G,A,2.0,stop_gained,HIGH,GLI3,ENSG00000106571,...,PERCENTILE:0.848408180476492,0.195,0.898291,YES,GGCCGCCCCGGTCAGCAGATGCTTG,1,CAG,TAG,"(CAG, TAG)",benign
1133,chr17,72124351,rs2143258813,G,A,2.0,stop_gained,HIGH,SOX9,ENSG00000125398,...,PERCENTILE:0.976470588235294,0.168,1.000000,YES,CCCCCAGCACTGGGAACAACCCGTC,3,TGG,TGA,"(TGG, TGA)",benign
